# 04 — Viz (exploratory)
Charts built on the normalized metrics from `03-prepare`. All comparisons use **share of series points**, so series of different lengths are on the same footing. PNGs land in `outputs/`.

Charts:
1. **Stacked share bar** — one bar per series (0–100%), segments are contestants by their share of series points.
2. **All contestants ranked** — every one of the 105 contestants ranked by `share_vs_equal`, best on top.
3. **Winners ranked** — the 21 series champions ranked by how dominant their win was.
4. **Winner vs runner-up gap** — how close each series was at the top.

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

import pandas as pd
import matplotlib.pyplot as plt
from src.ingest import load_config
from src.clean_quality import get_connection
from src.viz import (
    stacked_share_bar, all_contestants_ranked, winners_ranked, save_fig,
)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Load the prepared metrics
Reads the processed Parquet (falls back to the DuckDB table).

In [ ]:
proc = Path(cfg['paths']['data_processed']) / 'contestant_metrics.parquet'
if proc.exists():
    df = pd.read_parquet(proc)
else:
    df = con.execute('SELECT * FROM contestant_metrics').df()
print(df.shape)
df.head()

## 1. Stacked share bar — series on y, contestants as segments
Every bar spans 0–100%, so a 5-episode series and a 10-episode series are directly comparable. Segments are ordered best→worst left→right.

In [ ]:
fig = stacked_share_bar(df)
save_fig(fig, cfg, 'series_points_share_stacked')

## 2. All 105 contestants ranked (best on top)
Ranked by `share_vs_equal`: a contestant's share of series points relative to an even 1/5 split. 1.0 (dashed line) = exactly the average contestant that series. This is the apples-to-apples ranking across all series.

In [ ]:
fig = all_contestants_ranked(df, value_col='share_vs_equal')
save_fig(fig, cfg, 'all_contestants_ranked')

## 3. Series winners ranked by dominance
Of the 21 champions, who won their series most decisively (largest share of the series' points)?

In [ ]:
fig = winners_ranked(df)
save_fig(fig, cfg, 'winners_ranked_by_dominance')

## 4. How close was each series at the top?
Winner's share minus runner-up's share (in points of share). Small bars = nail-biters; big bars = runaways.

In [ ]:
from src.viz import _STYLE, _title_and_source

# Winner share minus runner-up share, per series.
top2 = (df[df['series_rank'].isin([1, 2])]
        .sort_values(['series', 'series_rank']))
gap = (top2.groupby('series')
            .agg(winner=('contestant', 'first'),
                 winner_share=('pct_of_series_points', 'first'),
                 runnerup_share=('pct_of_series_points', 'last'))
            .reset_index())
gap['margin'] = (gap['winner_share'] - gap['runnerup_share']) * 100
gap = gap.sort_values('margin', ascending=True)
labels = [f"{r.winner}  (S{r.series})" for r in gap.itertuples()]

with plt.rc_context(_STYLE):
    fig, ax = plt.subplots(figsize=(11, 9), dpi=150)
    bars = ax.barh(range(len(gap)), gap['margin'], color='#005F73')
    ax.bar_label(bars, fmt='%.1f pts', padding=3, fontsize=8)
    ax.set_yticks(range(len(gap)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlabel('Winner’s lead over runner-up (share-of-points percentage points)')
    ax.grid(axis='x', alpha=0.25); ax.grid(axis='y', visible=False)
    for sp in ('top', 'right', 'left'):
        ax.spines[sp].set_visible(False)
    _title_and_source(
        fig, ax,
        'Runaways and nail-biters',
        'Gap between the series winner and runner-up, in share-of-points percentage points',
        'Source: Taskmaster UK series 1–21, contestant point totals',
    )
save_fig(fig, cfg, 'winner_vs_runnerup_margin')

---
**Notes**
- These are exploratory. Curated, publication-ready social charts belong in `06-viz-social.ipynb` (`outputs/social/`, `twitter_landscape`, Pillow/chart_factory).
- Reminder caveat for any published version: points are a subjective, comedic score, and raw totals differ by series length — always compare on share.

---
## Cleanup

In [ ]:
con.close()
print('connection closed')